In [ ]:
!pip3.9 install --upgrade pip
!pip install --upgrade certifi
!pip install nltk
!pip install pymorphy2

In [3]:
import numpy as np
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import pandas as pd
from collections import Counter
import inspect
from nltk.corpus import stopwords
from pymorphy2 import MorphAnalyzer
import inspect



In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:

#Очистка, лемматизация и удаление стоп‑слов (рус). Возвращает строку токенов, содержащих только кириллицу.
MORPH = MorphAnalyzer()
def preprocess(text,
               _morph=MORPH,
               _en_stop=ENGLISH_STOP_WORDS,
               _ru_stop=set(stopwords.words('russian'))):
    """
    Очистка, лемматизация и удаление стоп‑слов (рус/англ).
    Возвращает строку токенов.
    """
    text = str(text).lower().replace('ё', 'е')
    tokens = re.findall(r'[a-zа-я]+', text)

    cleaned = []
    for tok in tokens:
        # английское слово
        if re.fullmatch(r'[a-z]+', tok):
            if tok not in _en_stop:
                cleaned.append(tok)
        # русское слово
        else:
            if tok in _ru_stop:
                continue
            lemma = _morph.parse(tok)[0].normal_form
            lemma = lemma.replace('ё', 'е')
            if lemma not in _ru_stop:
                cleaned.append(lemma)

    return ' '.join(cleaned)




#Количество комментариев по возрастам
def Count_comm_len(df, age_col='age'):
    counts = df[age_col].value_counts().sort_index()
    for age, cnt in counts.items():
        print(f'Возраст {age}: комментариев {cnt}')




#Строки с повтором от 4 раз
def split_rows_with_consecutive_repeats(df, text_col='comment', min_repeats=4):

    def has_consecutive_repeat(text):
        tokens = re.findall(r'\w+', str(text).lower())
        n = len(tokens)
        max_len = n // min_repeats
        for L in range(1, max_len + 1):
            for i in range(0, n - L * min_repeats + 1):
                phrase = tokens[i:i+L]
                if all(tokens[i + j*L : i + (j+1)*L] == phrase for j in range(min_repeats)):
                    return True
        return False

    mask = df[text_col].apply(has_consecutive_repeat)
    df_clean = df.loc[~mask].reset_index(drop=True)
    df_del   = df.loc[ mask].reset_index(drop=True)
    return df_clean, df_del


In [ ]:

df = pd.read_csv('data.csv')
df.count()


In [ ]:

# удаляем строки с повторами слов от 4 раз подряд
df, df_del = split_rows_with_consecutive_repeats(df, text_col='comment', min_repeats=3)
print(f"Оставшихся строк: {len(df)}, удалённых: {len(df_del)}")


df_del.head()

In [ ]:


#предобработка
df['comment'] = df['comment'].apply(preprocess)

df.head()


In [ ]:


#обрезка до диапозона слов
min_len = 15
max_len = 100

lengths = df['comment'].str.split().str.len()
df = df[lengths.between(min_len, max_len)].copy()


In [ ]:

#делим на интервалы возрастов
def age_to_group(age, bin_size=12, min_age=14):

    start = min_age + bin_size * ((age - min_age) // bin_size)
    end   = start + bin_size - 1
    return f"{start}-{end}"

# Применяем к столбцу:
df['age'] = df['age'].apply(age_to_group)



In [ ]:

Count_comm_len(df)

In [ ]:
# балансировка по минимальному размеру группы
min_count = df['age'].value_counts().min()
df = (
    df
    .groupby('age', group_keys=False)
    .apply(lambda g: g.sample(min_count, random_state=42))
    .reset_index(drop=True)
)

In [ ]:
Count_comm_len(df)

In [ ]:
df.head()

In [ ]:
from rapidfuzz import fuzz
from tqdm import tqdm

print(f"До дедупликации: {len(df)} строк")

from rapidfuzz import fuzz
from tqdm import tqdm

def remove_fuzzy_duplicates(df, text_col='comment', threshold=90):
    kept_texts = []
    keep_mask  = []

    for text in tqdm(df[text_col].astype(str), desc="Deduplicating"):
        if any(fuzz.ratio(text, kept) >= threshold for kept in kept_texts):
            keep_mask.append(False)
        else:
            kept_texts.append(text)
            keep_mask.append(True)

    return df.loc[keep_mask].reset_index(drop=True)

print(f"До дедупликации: {len(df)} строк") 

df = remove_fuzzy_duplicates(df, text_col='comment', threshold=90)
print(f"После дедупликации: {len(df)} строк")


In [17]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score


In [ ]:



# 1. Загрузка и разбиение данных
X = df[['comment']]
y = df['age']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


# 2. TF–IDF-препроцессор
preprocessor = ColumnTransformer(
    transformers=[
        ('txt', TfidfVectorizer(min_df=3, ngram_range=(1, 2)), 'comment')
    ],
    remainder='drop'
)


# 3. Модели
models = {
    'Logistic Regression': LogisticRegression(
        solver='saga',
        multi_class='multinomial',
        max_iter=500,
        n_jobs=-1
    ),
    'Multinomial NB'   : MultinomialNB(),
    'Linear SVC'       : LinearSVC(max_iter=5000, dual=False),
    'Random Forest'    : RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN'              : KNeighborsClassifier(n_jobs=-1, weights='distance')
}

# 4.Сетка Гиперпараметров
param_grids = {
    'Logistic Regression': {
        'clf__C'       : [0.1, 1, 5],
        'clf__penalty' : ['l2', 'elasticnet'],
        'clf__l1_ratio': [0.3, 0.5, 0.7]
    },

    'Multinomial NB': {
        'clf__alpha': [0.1, 0.5, 1.0]
    },

    'Linear SVC': {
        'clf__C'   : [0.5, 1, 2],
        'clf__loss': ['squared_hinge']
    },

    'Random Forest': {
        'clf__n_estimators'     : [200, 400],
        'clf__max_depth'        : [None, 20, 40],
        'clf__min_samples_split': [2, 5]
    },

    'Gradient Boosting': {
        'clf__n_estimators' : [200, 400],
        'clf__learning_rate': [0.05, 0.1],
        'clf__max_depth'    : [2, 3]
    },

    'KNN': {
        'clf__n_neighbors': [3, 5, 7],
        'clf__weights'    : ['uniform', 'distance']
    }
}

# 5. Перебор и отчёты
for name, model in models.items():
    print(f'\n=== {name} ===')

    pipeline = Pipeline([
        ('prep', preprocessor),
        ('clf', model)
    ])

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring='accuracy',
        cv=3,
        n_jobs=-1,
        refit=False
    )
    grid.fit(X_train, y_train)

    #результат каждой комбинации
    means, stds, params = (
        grid.cv_results_['mean_test_score'],
        grid.cv_results_['std_test_score'],
        grid.cv_results_['params']
    )

    for mean, std, p in zip(means, stds, params):
        print(f'\nПараметры: {p}')
        print(f'CV Accuracy: {mean:.3f} ± {std:.3f}')

        pipeline.set_params(**p)
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.3f}')
        print(classification_report(y_test, y_pred))

    #лучшая конфигурация
    best_idx = grid.best_index_
    best_params = params[best_idx]
    best_score  = means[best_idx]
    print(f'\n>>> Лучшие параметры (CV): {best_params}, CV Accuracy: {best_score:.3f}')

    print('\nОтчёт на тесте для лучших параметров:')
    pipeline.set_params(**best_params)
    pipeline.fit(X_train, y_train)
    y_best = pipeline.predict(X_test)
    print(f'Test Accuracy: {accuracy_score(y_test, y_best):.3f}')
    print(classification_report(y_test, y_best))


In [ ]:

from sklearn.ensemble import VotingClassifier


# Загрузка и разбиение данных
X = df[['comment']]
y = df['age']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


# TF–IDF-препроцессор
preprocessor = ColumnTransformer(
    transformers=[
        ('txt', TfidfVectorizer(min_df=3, ngram_range=(1, 2)), 'comment')
    ],
    remainder='drop'
)










# Создание ансамблевого классификатора

# Создание экземпляров моделей
log_reg = LogisticRegression(C=1, penalty='elasticnet', l1_ratio=0.5, solver='saga', multi_class='multinomial', max_iter=500, n_jobs=-1)
mnb = MultinomialNB(alpha=0.5)
svc = LinearSVC(C=1, loss='squared_hinge', max_iter=5000, dual=True)

# Создание ансамбля с голосованием (hard voting)
ensemble_model = VotingClassifier(
    estimators=[
        ('log_reg', log_reg),
        ('mnb', mnb),
        ('svc', svc)
    ],
    voting='hard'  # Голосование по большинству
)


# Обучение и оценка ансамбля

pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', ensemble_model)
])

# Обучение модели
pipeline.fit(X_train, y_train)

# Предсказание на тестовых данных
y_pred = pipeline.predict(X_test)

# Оценка качества модели
print(f'Ensemble Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(classification_report(y_test, y_pred))


In [ ]:
import joblib
joblib.dump(pipeline, 'age_classifier_pipeline.pkl')
